In [1]:
!pip install -q "transformers>=4.51.0" datasets torch accelerate peft sentencepiece protobuf openai


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import re
import os
import random
import time
import gc
from datetime import datetime, timedelta
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup, Adafactor
from peft import LoraConfig, get_peft_model, TaskType
from openai import OpenAI

# ============================================================
# CONFIGURATION
# ============================================================
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME   = "Qwen/Qwen3-1.7B"
MAX_LENGTH   = 1024
MAX_NEW_TOKENS = 2048

# Training hyperparameters
LEARNING_RATE    = 2e-5
WEIGHT_DECAY     = 0.01
LORA_RANK        = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.1
NUM_TRAIN        = 1000
NUM_TEST         = 100
NUM_EPOCHS       = 2
GRAD_ACCUM_STEPS = 8
MAX_GRAD_NORM    = 1.0
WARMUP_RATIO     = 0.1

# ---- Fine-tuning mode ----
USE_LORA = False   # Set to False for full fine-tuning (no PEFT)

# Concept-vector construction
NUM_CONCEPT_SAMPLES = 50
CONCEPT_MAX_ATTEMPTS = 250
DAILYDIALOG_CONFIG = "full"
CONCEPT_LAYER = -1

# Composite loss weights
GAMMA   = 1.0   # CE loss weight
LAMBDA_ = 0.1   # Concept vector cosine distance weight

# Language
LANG_CONFIG = "pa"
LANG_NAME   = "Punjabi"

# OpenRouter API for DailyDialog → Gurmukhi Punjabi translation
OPENROUTER_API_KEY = "YOUR_KEY_HERE"   # <-- SET THIS
PA_DAILYDIALOG_CACHE = "pa_dailydialog_cache.json"

TRANSLATION_SYSTEM_PROMPT = (
    "You are an expert Punjabi translator. Translate the given English text "
    "to Punjabi in Gurmukhi script (ਗੁਰਮੁਖੀ).\n"
    "Rules:\n"
    "1. Use ONLY Gurmukhi script — never Shahmukhi or Roman/Latin letters\n"
    "2. Sound completely natural and colloquial — like a native speaker "
    "from Punjab, India would say it in everyday conversation\n"
    "3. Do NOT sound mechanical or robotic — avoid literal word-for-word translation\n"
    "4. Match the tone of the original (casual, formal, humorous, etc.)\n"
    "5. Output ONLY the Punjabi translation — no explanations, no English, no notes"
)

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

PROMPT_TEMPLATE = (
    "Solve this math problem. Show your work step by step "
    "and give the final numeric answer.\n\nQuestion: {question}"
)

# Derived
steps_per_epoch = NUM_TRAIN // GRAD_ACCUM_STEPS
total_steps     = steps_per_epoch * NUM_EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)

print("=" * 70)
print("  Concept Vector Alignment SFT on Qwen3-1.7B")
print("=" * 70)
print(f"  Model:                {MODEL_NAME}")
print(f"  Language:             {LANG_NAME} ({LANG_CONFIG})")
print(f"  Train / Test:         {NUM_TRAIN} / {NUM_TEST}")
print(f"  Epochs:               {NUM_EPOCHS}")
print(f"  Grad accumulation:    {GRAD_ACCUM_STEPS}")
print(f"  Effective batch size: {GRAD_ACCUM_STEPS}")
print(f"  Steps per epoch:      {steps_per_epoch}")
print(f"  Total optim. steps:   {total_steps}")
print(f"  Warmup steps:         {warmup_steps}")
print(f"  LR:                   {LEARNING_RATE}")
print(f"  Loss:                 {GAMMA}·CE + {LAMBDA_}·(1 − CosSim)")
print(f"  Concept samples:      {NUM_CONCEPT_SAMPLES} (paired EN↔PA)")
if USE_LORA:
    print(f"  LoRA:                 rank={LORA_RANK} α={LORA_ALPHA} drop={LORA_DROPOUT}")
else:
    print(f"  Fine-tuning:          FULL (all parameters, no LoRA)")
print(f"  Concept vec recompute: every optimizer step (with grad)")
print("=" * 70)


In [5]:
# ============================================================
# LOAD MODEL + (optional LoRA)
# ============================================================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    use_cache=False,
)

if USE_LORA:
    print("Applying LoRA adapters...")
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )
    model = get_peft_model(model, lora_config)
    model.enable_input_require_grads()
else:
    print("Full fine-tuning mode (no LoRA)")

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.2f}%)")

device = next(model.parameters()).device
print(f"Device: {device}")
print(f"Dtype:  {next(model.parameters()).dtype}")


In [6]:
# ============================================================
# LOAD PAIRED DATASET (sarvamai/gsm8k-indic, Punjabi config)
# ============================================================
print(f"Loading sarvamai/gsm8k-indic  config={LANG_CONFIG}...")
indic_ds = load_dataset("sarvamai/gsm8k-indic", LANG_CONFIG, split="test")
print(f"Total samples: {len(indic_ds)}")
print(f"Fields: {indic_ds.column_names}")

all_indices = list(range(len(indic_ds)))
random.Random(SEED).shuffle(all_indices)

train_indices   = all_indices[:NUM_TRAIN]
test_indices    = all_indices[NUM_TRAIN : NUM_TRAIN + NUM_TEST]
concept_indices = all_indices[NUM_TRAIN + NUM_TEST:]
assert len(set(train_indices) & set(test_indices)) == 0, "Train-test leak!"

train_samples = []
for idx in train_indices:
    r = indic_ds[idx]
    train_samples.append({
        "pa_question": r["question"],
        "pa_answer":   r["answer"],
        "en_question":  r["original_question"],
        "en_answer":    r["original_answer"],
    })

test_samples = []
for idx in test_indices:
    r = indic_ds[idx]
    test_samples.append({
        "pa_question": r["question"],
        "pa_answer":   r["answer"],
        "en_question":  r["original_question"],
        "en_answer":    r["original_answer"],
    })

print(f"Train:   {len(train_samples)} paired (PA↔EN) samples")
print(f"Test:    {len(test_samples)} samples (unseen)")
print(f"Concept: {len(concept_indices)} samples (non-overlapping pool)")

print("\n--- Pairing verification (first 3) ---")
for i in range(3):
    s = train_samples[i]
    en_num = re.search(r'####\s*(-?[\d,]+\.?\d*)', s["en_answer"])
    pa_num = re.search(r'####\s*(-?[\d,]+\.?\d*)', s["pa_answer"])
    en_n = en_num.group(1) if en_num else "?"
    pa_n = pa_num.group(1) if pa_num else "?"
    print(f"[{i}] EN: {s['en_question'][:60]}…  →  {en_n}")
    print(f"    PA: {s['pa_question'][:60]}…  →  {pa_n}")
    print(f"    Match: {en_n == pa_n}")


In [7]:
# ============================================================
# HELPERS
# ============================================================

def extract_gold_answer(answer_str):
    """Extract numeric answer from GSM8K '#### N' format."""
    m = re.search(r'####\s*(-?[\d,]+\.?\d*)', answer_str)
    return m.group(1).replace(",", "").strip() if m else None

def extract_model_answer(text):
    """Extract last numeric value from model output."""
    boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        n = re.sub(r'[^\d.\-]', '', boxed[-1])
        if n: return n
    for pat in [
        r'(?:answer|result|total)\s*(?:is|=|:)\s*(-?[\d,]+\.?\d*)',
        r'####\s*(-?[\d,]+\.?\d*)',
        r'=\s*(-?[\d,]+\.?\d*)\s*$',
    ]:
        ms = re.findall(pat, text, re.I | re.M)
        if ms: return ms[-1].replace(",", "").strip()
    nums = re.findall(r'-?[\d,]+\.?\d*', text)
    return nums[-1].replace(",", "").strip() if nums else None

def compare_answers(pred, gold):
    if pred is None or gold is None: return False
    try:    return abs(float(pred) - float(gold)) < 1e-3
    except: return pred.strip() == gold.strip()


def build_prompts(sample):
    """Return (pa_full_text, pa_q_only_text) for CE loss + label masking."""
    pa_q = PROMPT_TEMPLATE.format(question=sample["pa_question"])

    pa_full_text = tokenizer.apply_chat_template(
        [{"role": "user",      "content": pa_q},
         {"role": "assistant", "content": sample["pa_answer"]}],
        tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )

    pa_q_only_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": pa_q}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

    return pa_full_text, pa_q_only_text


def format_gsm8k_sft(question, answer):
    """Format GSM8K Q+A as SFT text (same prompt template for EN and PA)."""
    q = PROMPT_TEMPLATE.format(question=question)
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}, {"role": "assistant", "content": answer}],
        tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )


def format_dailydialog_sft(question, answer):
    """Format DailyDialog Q+A as SFT text (same prompt for EN and PA)."""
    q = f"Respond naturally to this dialogue line:\n\n{question}"
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": q}, {"role": "assistant", "content": answer}],
        tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )


def generate_and_grade(question, gold_answer):
    """Generate answer and grade correctness."""
    prompt = PROMPT_TEMPLATE.format(question=question)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)

    with torch.no_grad():
        gen_ids = model.generate(
            **enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    continuation = gen_ids[0, enc["input_ids"].shape[1]:]
    decoded = tokenizer.decode(continuation, skip_special_tokens=True)
    pred = extract_model_answer(decoded)
    hit = compare_answers(pred, gold_answer)
    return hit, pred, decoded


def extract_last_hidden(text, model_obj):
    """Extract last-token hidden state (detached, CPU, float32)."""
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
    out = model_obj(**enc, output_hidden_states=True)
    h = out.hidden_states[CONCEPT_LAYER][0, -1, :].detach().float().cpu()
    del out
    return h


# --- OpenRouter translation ---
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

def translate_text_to_punjabi(text, max_retries=3):
    """Translate English text to Gurmukhi Punjabi via OpenRouter GPT-5.4."""
    for attempt in range(max_retries):
        try:
            resp = openrouter_client.chat.completions.create(
                model="openai/gpt-5.4",
                messages=[
                    {"role": "system", "content": TRANSLATION_SYSTEM_PROMPT},
                    {"role": "user", "content": f"Translate to Punjabi (Gurmukhi script):\n\n{text}"}
                ],
                temperature=0.3,
                max_tokens=1024,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt + random.random()
            print(f"  Translation API error (attempt {attempt+1}): {e}. Retrying in {wait:.1f}s...")
            time.sleep(wait)


def compute_concept_loss_with_grad(model_obj, pa_gsm8k_texts, pa_dd_texts,
                                   en_cv_frozen, lambda_val):
    """Compute PA concept vector WITH gradients, backward the concept loss.

    Returns (cosine_sim_value, concept_loss_value).
    Gradients are accumulated onto model parameters.
    """
    was_training = model_obj.training
    model_obj.eval()  # disable dropout for stable concept vector

    gsm8k_hidden = []
    for text in pa_gsm8k_texts:
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
        out = model_obj(**enc, output_hidden_states=True)
        h = out.hidden_states[CONCEPT_LAYER][0, -1, :].float()
        gsm8k_hidden.append(h)
        del out, enc

    dd_hidden = []
    for text in pa_dd_texts:
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
        out = model_obj(**enc, output_hidden_states=True)
        h = out.hidden_states[CONCEPT_LAYER][0, -1, :].float()
        dd_hidden.append(h)
        del out, enc

    pa_cv = torch.stack(gsm8k_hidden).mean(dim=0) - torch.stack(dd_hidden).mean(dim=0)

    pa_cv_norm = F.normalize(pa_cv, dim=0)
    cos_sim = torch.dot(pa_cv_norm, en_cv_frozen.float())
    concept_loss = lambda_val * (1.0 - cos_sim)
    concept_loss.backward()

    cos_val = cos_sim.item()
    concept_val = concept_loss.item()

    del pa_cv, pa_cv_norm, cos_sim, concept_loss, gsm8k_hidden, dd_hidden
    torch.cuda.empty_cache()

    if was_training:
        model_obj.train()

    return cos_val, concept_val


# ---- Verification ----
s0 = train_samples[0]
_pa_full, _pa_q = build_prompts(s0)
_pa_full_ids = tokenizer.encode(_pa_full)
_pa_q_ids    = tokenizer.encode(_pa_q)
assert _pa_full_ids[:len(_pa_q_ids)] == _pa_q_ids, "PA question prefix mismatch!"

print("✓ Prompt structure verified.")
print(f"  PA Q-only tokens      : {len(_pa_q_ids)}")
print(f"  PA full (Q+A) tokens  : {len(_pa_full_ids)}")
print(f"  Answer tokens (PA)    : {len(_pa_full_ids) - len(_pa_q_ids)}")


In [8]:
# ============================================================
# PREPARE CONCEPT SAMPLES + BUILD CONCEPT VECTORS
# ============================================================

# ---- 1. Find paired GSM8K concept samples (EN+PA both correct) ----
print("Finding paired GSM8K concept samples (EN+PA both correct)...")
print(f"  Pool: {len(concept_indices)} | Target: {NUM_CONCEPT_SAMPLES}")

model.eval()
concept_gsm8k_pairs = []   # (en_q, en_a, pa_q, pa_a)
concept_attempted = 0
concept_en_fail = 0
concept_pa_fail = 0
gsm_start = time.time()

with torch.no_grad():
    for idx in concept_indices:
        if len(concept_gsm8k_pairs) >= NUM_CONCEPT_SAMPLES:
            break
        if concept_attempted >= CONCEPT_MAX_ATTEMPTS:
            break

        row = indic_ds[idx]
        en_q, en_a = row["original_question"], row["original_answer"]
        pa_q, pa_a = row["question"], row["answer"]
        en_gold = extract_gold_answer(en_a)
        pa_gold = extract_gold_answer(pa_a)
        concept_attempted += 1

        if en_gold is None or pa_gold is None:
            continue

        en_hit, _, _ = generate_and_grade(en_q, en_gold)
        if not en_hit:
            concept_en_fail += 1
            continue

        pa_hit, _, _ = generate_and_grade(pa_q, pa_gold)
        if not pa_hit:
            concept_pa_fail += 1
            continue

        concept_gsm8k_pairs.append((en_q, en_a, pa_q, pa_a))

        if concept_attempted % 10 == 0 or len(concept_gsm8k_pairs) == NUM_CONCEPT_SAMPLES:
            elapsed = time.time() - gsm_start
            print(f"  {len(concept_gsm8k_pairs)}/{NUM_CONCEPT_SAMPLES} pairs, "
                  f"attempted {concept_attempted}, {elapsed:.0f}s")

if len(concept_gsm8k_pairs) < NUM_CONCEPT_SAMPLES:
    print(f"  WARNING: Only {len(concept_gsm8k_pairs)} pairs found. Adjusting.")
    NUM_CONCEPT_SAMPLES = len(concept_gsm8k_pairs)

print(f"\n✓ {NUM_CONCEPT_SAMPLES} paired GSM8K concept samples")
print(f"  attempted={concept_attempted}  EN_fail={concept_en_fail}  PA_fail={concept_pa_fail}")

en_gsm8k_concept_texts = [format_gsm8k_sft(eq, ea) for (eq, ea, _, _) in concept_gsm8k_pairs]
pa_gsm8k_concept_texts = [format_gsm8k_sft(pq, pa) for (_, _, pq, pa) in concept_gsm8k_pairs]

# ---- 2. Load DailyDialog ----
print("\n--- Loading DailyDialog ---")

def load_dailydialog_rows_via_server(max_rows=12000, page_size=50, max_retries=8,
                                     target_pairs=1000, pair_buffer=250):
    import urllib.parse, urllib.request, urllib.error
    rows, offset = [], 0
    fetch_start = time.time()
    target_pair_budget = target_pairs + pair_buffer
    running_pair_estimate = 0

    while offset < max_rows:
        length = min(page_size, max_rows - offset)
        url = ("https://datasets-server.huggingface.co/rows?"
               + urllib.parse.urlencode({
                   "dataset": "roskoN/dailydialog", "config": DAILYDIALOG_CONFIG,
                   "split": "train", "offset": offset, "length": length}))
        payload = None
        for attempt in range(max_retries):
            try:
                with urllib.request.urlopen(url, timeout=30) as resp:
                    payload = json.loads(resp.read().decode("utf-8"))
                break
            except urllib.error.HTTPError as e:
                if e.code != 429 or attempt == max_retries - 1: raise
                time.sleep(min(90, 2 ** attempt + random.random()))
            except urllib.error.URLError:
                if attempt == max_retries - 1: raise
                time.sleep(min(30, 2 ** attempt + random.random()))

        page_rows = payload.get("rows", []) if payload else []
        if not page_rows: break
        for item in page_rows:
            row = item.get("row", {})
            utt = row.get("utterances")
            if utt:
                rows.append({"utterances": utt})
                running_pair_estimate += max(0, len(utt) - 1)
        offset += len(page_rows)
        if running_pair_estimate >= target_pair_budget: break
        if len(page_rows) < length: break
    return rows

try:
    dailydialog = load_dataset("roskoN/dailydialog", DAILYDIALOG_CONFIG, split="train")
    dailydialog_rows = dailydialog
except RuntimeError as e:
    if "Dataset scripts are no longer supported" not in str(e): raise
    print("  Using dataset-server rows API.")
    dailydialog_rows = load_dailydialog_rows_via_server(
        max_rows=11118, target_pairs=NUM_CONCEPT_SAMPLES,
        pair_buffer=max(200, NUM_CONCEPT_SAMPLES // 4))

dd_pairs = []
for row in dailydialog_rows:
    utt = row["utterances"]
    for i in range(len(utt) - 1):
        q, a = utt[i].strip(), utt[i + 1].strip()
        if q and a:
            dd_pairs.append((q, a))

if len(dd_pairs) < NUM_CONCEPT_SAMPLES:
    raise ValueError(f"DailyDialog: only {len(dd_pairs)} pairs (need {NUM_CONCEPT_SAMPLES})")

# Deterministic shuffle independent of earlier random state
dd_rng = random.Random(SEED + 100)
dd_rng.shuffle(dd_pairs)
dd_selected = dd_pairs[:NUM_CONCEPT_SAMPLES]

# ---- 3. Translate DailyDialog to Gurmukhi Punjabi (cached) ----
print(f"\n--- Translating {NUM_CONCEPT_SAMPLES} DailyDialog pairs → Gurmukhi Punjabi ---")

if os.path.exists(PA_DAILYDIALOG_CACHE):
    with open(PA_DAILYDIALOG_CACHE, "r", encoding="utf-8") as f:
        pa_dd_cache = json.load(f)
    # Validate cache matches current selection
    cache_valid = all(
        pa_dd_cache[j]["en_q"] == dd_selected[j][0]
        for j in range(min(len(pa_dd_cache), len(dd_selected)))
    ) if pa_dd_cache else True
    if not cache_valid:
        print("  Cache mismatch detected — re-translating from scratch.")
        pa_dd_cache = []
    else:
        print(f"  Loaded {len(pa_dd_cache)} cached translations")
else:
    pa_dd_cache = []

translate_start = time.time()
for j in range(len(pa_dd_cache), NUM_CONCEPT_SAMPLES):
    en_q, en_a = dd_selected[j]
    pa_q = translate_text_to_punjabi(en_q)
    pa_a = translate_text_to_punjabi(en_a)
    pa_dd_cache.append({"en_q": en_q, "en_a": en_a, "pa_q": pa_q, "pa_a": pa_a})
    # Save after every translation (crash-safe)
    with open(PA_DAILYDIALOG_CACHE, "w", encoding="utf-8") as f:
        json.dump(pa_dd_cache, f, ensure_ascii=False, indent=2)
    if (j + 1) % 10 == 0 or j + 1 == NUM_CONCEPT_SAMPLES:
        print(f"  Translated {j+1}/{NUM_CONCEPT_SAMPLES} ({time.time()-translate_start:.0f}s)")

en_dd_concept_texts = [format_dailydialog_sft(q, a) for (q, a) in dd_selected]
pa_dd_concept_texts = [format_dailydialog_sft(e["pa_q"], e["pa_a"])
                       for e in pa_dd_cache[:NUM_CONCEPT_SAMPLES]]

print(f"\n✓ Concept sample preparation complete")
print(f"  GSM8K:      {len(en_gsm8k_concept_texts)} EN, {len(pa_gsm8k_concept_texts)} PA (sample-aligned)")
print(f"  DailyDialog: {len(en_dd_concept_texts)} EN, {len(pa_dd_concept_texts)} PA (translated)")
print(f"\n  Sample translation:")
print(f"    EN Q: {dd_selected[0][0][:80]}")
print(f"    PA Q: {pa_dd_cache[0]['pa_q'][:80]}")

# ---- 4. Build English concept vector (FROZEN) ----
print("\n--- Building English concept vector (frozen, static) ---")
en_gsm8k_states, en_dd_states = [], []
en_start = time.time()
with torch.no_grad():
    for j, text in enumerate(en_gsm8k_concept_texts, 1):
        en_gsm8k_states.append(extract_last_hidden(text, model))
        if j % 25 == 0: print(f"  EN GSM8K: {j}/{NUM_CONCEPT_SAMPLES}")
    for j, text in enumerate(en_dd_concept_texts, 1):
        en_dd_states.append(extract_last_hidden(text, model))
        if j % 25 == 0: print(f"  EN DailyDialog: {j}/{NUM_CONCEPT_SAMPLES}")

en_concept_vec = torch.stack(en_gsm8k_states).mean(0) - torch.stack(en_dd_states).mean(0)
en_concept_vec_frozen = F.normalize(en_concept_vec, dim=0).to(device).detach().clone()
print(f"  ✓ ||en_cv||: {en_concept_vec.norm():.4f}")

# ---- 5. Initial Punjabi concept vector (pre-training baseline) ----
print("\n--- Building initial Punjabi concept vector ---")
pa_gsm8k_states, pa_dd_states = [], []
with torch.no_grad():
    for j, text in enumerate(pa_gsm8k_concept_texts, 1):
        pa_gsm8k_states.append(extract_last_hidden(text, model))
        if j % 25 == 0: print(f"  PA GSM8K: {j}/{NUM_CONCEPT_SAMPLES}")
    for j, text in enumerate(pa_dd_concept_texts, 1):
        pa_dd_states.append(extract_last_hidden(text, model))
        if j % 25 == 0: print(f"  PA DailyDialog: {j}/{NUM_CONCEPT_SAMPLES}")

pa_cv_initial = torch.stack(pa_gsm8k_states).mean(0) - torch.stack(pa_dd_states).mean(0)
pa_cv_norm_initial = F.normalize(pa_cv_initial, dim=0)
baseline_cos_sim = torch.dot(pa_cv_norm_initial, en_concept_vec_frozen.cpu().float()).item()

print(f"  ✓ ||pa_cv||: {pa_cv_initial.norm():.4f}")
print(f"\n  Pre-training CosSim(PA_cv, EN_cv): {baseline_cos_sim:.4f}")

concept_stats = {
    "num_concept_samples": NUM_CONCEPT_SAMPLES,
    "concept_attempted": concept_attempted,
    "en_fail": concept_en_fail, "pa_fail": concept_pa_fail,
    "concept_layer": CONCEPT_LAYER,
    "en_cv_norm": float(en_concept_vec.norm().item()),
    "pa_cv_norm_initial": float(pa_cv_initial.norm().item()),
    "baseline_cosine_sim": round(baseline_cos_sim, 6),
}


In [9]:
# ============================================================
# TRAINING LOOP — Concept Vector Alignment SFT
#   Per sample:  PA Q+A → CE loss (1 forward pass)
#   Per optim step: recompute PA concept vec → cosine loss (with grad)
# ============================================================
print("=" * 70)
print("  STARTING TRAINING")
print("  Loss: γ·CE + λ·(1 − CosSim(PA_cv, EN_cv_frozen))")
print("  PA concept vec recomputed every optimizer step WITH gradients")
print("=" * 70)

model.train()

optimizer = Adafactor(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    relative_step=False, scale_parameter=False, warmup_init=False,
)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
)

training_log = []
global_step = 0
accumulated = 0
optimizer.zero_grad()
total_start = time.time()
sample_times = []

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    epoch_ce, epoch_concept, epoch_cos, epoch_total = [], [], [], []

    epoch_samples = train_samples.copy()
    random.shuffle(epoch_samples)

    print(f"\n{'='*70}")
    print(f"  EPOCH {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*70}")

    for i, sample in enumerate(epoch_samples):
        t0 = time.time()

        # ---- CE FORWARD PASS ----
        pa_full_text, pa_q_only = build_prompts(sample)
        pa_q_len = len(tokenizer.encode(pa_q_only))

        pa_full_enc = tokenizer(pa_full_text, return_tensors="pt",
                                truncation=True, max_length=MAX_LENGTH).to(device)
        labels = pa_full_enc["input_ids"].clone()
        labels[0, :min(pa_q_len, labels.shape[1])] = -100

        outputs_ce = model(
            input_ids=pa_full_enc["input_ids"],
            attention_mask=pa_full_enc["attention_mask"],
            labels=labels,
        )
        ce_loss = outputs_ce.loss
        (GAMMA * ce_loss / GRAD_ACCUM_STEPS).backward()
        accumulated += 1

        ce_val = ce_loss.item()
        epoch_ce.append(ce_val)
        del outputs_ce, ce_loss, pa_full_enc, labels
        torch.cuda.empty_cache()

        # ---- OPTIMIZER STEP (every GRAD_ACCUM_STEPS) ----
        cos_val_step = None
        concept_val_step = None

        if accumulated >= GRAD_ACCUM_STEPS:
            # Recompute PA concept vector WITH gradients + backward
            cos_val_step, concept_val_step = compute_concept_loss_with_grad(
                model, pa_gsm8k_concept_texts, pa_dd_concept_texts,
                en_concept_vec_frozen, LAMBDA_
            )
            epoch_cos.append(cos_val_step)
            epoch_concept.append(concept_val_step)
            total_val = GAMMA * (sum(epoch_ce[-GRAD_ACCUM_STEPS:]) / GRAD_ACCUM_STEPS) + concept_val_step
            epoch_total.append(total_val)

            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            accumulated = 0
            global_step += 1

        dt = time.time() - t0
        sample_times.append(dt)

        training_log.append({
            "epoch": epoch + 1, "sample": i + 1,
            "global_sample": epoch * NUM_TRAIN + i + 1,
            "global_step": global_step,
            "ce_loss": round(ce_val, 6),
            "concept_loss": round(concept_val_step, 6) if concept_val_step is not None else None,
            "cosine_sim": round(cos_val_step, 6) if cos_val_step is not None else None,
            "lr": scheduler.get_last_lr()[0],
            "step_time_s": round(dt, 3),
        })

        if (i + 1) % 50 == 0 or i == 0 or (i + 1) == NUM_TRAIN:
            window = min(50, len(epoch_ce))
            avg_t = sum(sample_times[-window:]) / window
            rem = (NUM_EPOCHS - epoch) * NUM_TRAIN - (i + 1)
            eta = str(timedelta(seconds=int(rem * avg_t)))
            cos_s = f"{epoch_cos[-1]:.4f}" if epoch_cos else "—"
            con_s = f"{epoch_concept[-1]:.4f}" if epoch_concept else "—"
            print(
                f"  [{i+1:3d}/{NUM_TRAIN}] "
                f"CE:{sum(epoch_ce[-window:])/window:.4f}  "
                f"Concept:{con_s}  CosSim:{cos_s}  "
                f"LR:{scheduler.get_last_lr()[0]:.1e}  "
                f"{avg_t:.2f}s/samp  ETA:{eta}"
            )

        if (i + 1) % 100 == 0:
            ckpt = f"{RESULTS_DIR}/checkpoint_epoch_{epoch+1}_step_{i+1}"
            print(f"\n  Saving checkpoint to {ckpt}...")
            model.save_pretrained(ckpt)
            tokenizer.save_pretrained(ckpt)

    # Flush remaining accumulated gradients
    if accumulated > 0:
        cos_flush, concept_flush = compute_concept_loss_with_grad(
            model, pa_gsm8k_concept_texts, pa_dd_concept_texts,
            en_concept_vec_frozen, LAMBDA_
        )
        epoch_cos.append(cos_flush)
        epoch_concept.append(concept_flush)

        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        accumulated = 0; global_step += 1

    et = time.time() - epoch_start
    print(f"\n  ── Epoch {epoch+1} summary ──")
    print(f"     CE Loss     : {sum(epoch_ce)/len(epoch_ce):.4f}")
    if epoch_concept:
        print(f"     Concept Loss: {sum(epoch_concept)/len(epoch_concept):.4f}")
        print(f"     CosSim      : {sum(epoch_cos)/len(epoch_cos):.4f}")
    print(f"     Time        : {et:.0f}s  ({et/60:.1f} min)")
    print(f"     Opt. steps  : {global_step}")

    epoch_ckpt = f"{RESULTS_DIR}/checkpoint_epoch_{epoch+1}"
    print(f"\n  Saving checkpoint to {epoch_ckpt}...")
    model.save_pretrained(epoch_ckpt)
    tokenizer.save_pretrained(epoch_ckpt)

tt = time.time() - total_start
print(f"\n{'='*70}")
print(f"  TRAINING COMPLETE — {tt:.0f}s ({tt/60:.1f} min)")
print(f"  Total optimizer steps: {global_step}")
print(f"{'='*70}")


In [10]:
# ============================================================
# POST-TRAINING: recompute PA concept vector, compare with baseline
# ============================================================
print("Recomputing post-training Punjabi concept vector...")
model.eval()

post_pa_gsm8k, post_pa_dd = [], []
with torch.no_grad():
    for text in pa_gsm8k_concept_texts:
        post_pa_gsm8k.append(extract_last_hidden(text, model))
    for text in pa_dd_concept_texts:
        post_pa_dd.append(extract_last_hidden(text, model))

post_pa_cv = torch.stack(post_pa_gsm8k).mean(0) - torch.stack(post_pa_dd).mean(0)
post_pa_cv_norm = F.normalize(post_pa_cv, dim=0)
post_cos_sim = torch.dot(post_pa_cv_norm, en_concept_vec_frozen.cpu().float()).item()

print(f"\n{'':18s} {'Pre-Train':>12s} {'Post-Train':>12s} {'Δ':>10s}")
print("-" * 56)
print(f"  {'CosSim':12s}   {baseline_cos_sim:10.4f}   {post_cos_sim:10.4f}   {post_cos_sim-baseline_cos_sim:+8.4f}")
print(f"  {'||PA_cv||':12s}   {pa_cv_initial.norm():10.4f}   {post_pa_cv.norm():10.4f}")


In [11]:
# ============================================================
# EVALUATION — 100 unseen Punjabi samples
# ============================================================
print("=" * 70)
print("  EVALUATION: 100 unseen Punjabi GSM8K samples")
print("=" * 70)

model.gradient_checkpointing_disable()   # not needed for inference
model.eval()

eval_logs  = []
correct    = 0
eval_start = time.time()

for i, sample in enumerate(test_samples):
    question  = sample["pa_question"]
    gold_raw  = sample["pa_answer"]
    gold      = extract_gold_answer(gold_raw)
    if gold is None:
        gold = extract_gold_answer(sample["en_answer"])

    prompt  = PROMPT_TEMPLATE.format(question=question)
    msgs    = [{"role": "user", "content": prompt}]
    text    = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=MAX_LENGTH).to(device)

    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.7, top_p=0.8, top_k=20,
            do_sample=True,
        )
    out_ids  = gen[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(out_ids, skip_special_tokens=True).strip()
    pred     = extract_model_answer(response)
    hit      = compare_answers(pred, gold)
    if hit:
        correct += 1

    eval_logs.append({
        "index": i, "pa_question": question,
        "en_question": sample["en_question"],
        "gold": gold, "pred": pred, "correct": hit,
        "response_preview": response[:400],
    })

    if (i + 1) % 10 == 0 or i == 0:
        print(f"  [{i+1:3d}/{NUM_TEST}]  "
              f"Gold:{str(gold):>6s}  Pred:{str(pred):>6s}  "
              f"{'✓' if hit else '✗'}   "
              f"Running: {correct}/{i+1} = {correct/(i+1)*100:.1f}%")

eval_time = time.time() - eval_start
accuracy  = correct / NUM_TEST * 100

print(f"\n{'='*70}")
print(f"  RESULT: {correct}/{NUM_TEST} = {accuracy:.1f}%")
print(f"  Time:   {eval_time:.0f}s  ({eval_time/NUM_TEST:.1f}s / sample)")
print(f"{'='*70}")


In [12]:
# ============================================================
# SAVE EVERYTHING
# ============================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

summary = {
    "model": MODEL_NAME,
    "method": "Concept Vector Alignment SFT",
    "language": f"{LANG_NAME} ({LANG_CONFIG})",
    "timestamp": timestamp,
    "config": {
        "seed": SEED, "lr": LEARNING_RATE, "epochs": NUM_EPOCHS,
        "grad_accum": GRAD_ACCUM_STEPS, "max_length": MAX_LENGTH,
        "fine_tuning_mode": "LoRA" if USE_LORA else "Full",
        "lora_rank": LORA_RANK if USE_LORA else None,
        "lora_alpha": LORA_ALPHA if USE_LORA else None,
        "gamma_ce": GAMMA, "lambda_concept": LAMBDA_,
        "num_train": NUM_TRAIN, "num_test": NUM_TEST,
    },
    "concept_vector": concept_stats,
    "cosine_similarity": {
        "pre_training": round(baseline_cos_sim, 4),
        "post_training": round(post_cos_sim, 4),
        "delta": round(post_cos_sim - baseline_cos_sim, 4),
    },
    "eval": {
        "correct": correct, "total": NUM_TEST,
        "accuracy": round(accuracy, 2),
        "time_s": round(eval_time, 1),
    },
    "training_time_s": round(tt, 1),
}

all_data = {
    "summary": summary,
    "training_log": training_log,
    "eval_logs": eval_logs,
}

log_path = f"{RESULTS_DIR}/concept_vec_train_{timestamp}.json"
sum_path = f"{RESULTS_DIR}/concept_vec_summary_{timestamp}.json"

with open(log_path, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)
with open(sum_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"Logs saved    → {log_path}")
print(f"Summary saved → {sum_path}")

print(f"\n{'='*70}")
print(f"  FINAL REPORT — Concept Vec Alignment on {MODEL_NAME}")
print(f"{'='*70}")
print(f"  Language:          {LANG_NAME}")
print(f"  Training samples:  {NUM_TRAIN} × {NUM_EPOCHS} epochs")
print(f"  Training time:     {tt/60:.1f} min")
print(f"")
print(f"  Concept Vector Alignment (PA_cv ↔ EN_cv):")
print(f"    Pre-training:    {baseline_cos_sim:.4f}")
print(f"    Post-training:   {post_cos_sim:.4f}  (Δ = {post_cos_sim-baseline_cos_sim:+.4f})")
print(f"")
print(f"  Evaluation (100 unseen PA samples):")
print(f"    Accuracy:        {correct}/{NUM_TEST} = {accuracy:.1f}%")
print(f"{'='*70}")

if USE_LORA:
    adapter_path = f"{RESULTS_DIR}/lora_adapter_{timestamp}"
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    print(f"\nLoRA adapter saved → {adapter_path}/")
else:
    ft_path = f"{RESULTS_DIR}/full_ft_model_{timestamp}"
    model.save_pretrained(ft_path)
    tokenizer.save_pretrained(ft_path)
    print(f"\nFull fine-tuned model saved → {ft_path}/")


In [ ]:
# ============================================================
# BASELINE SFT: Load a FRESH model (no JEPA-trained weights)
# ============================================================
import gc

# Free JEPA model memory
del model, optimizer, scheduler
gc.collect()
torch.cuda.empty_cache()

print("Loading fresh model for baseline SFT...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    use_cache=False,
)

if USE_LORA:
    lora_config_bl = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )
    base_model = get_peft_model(base_model, lora_config_bl)
    base_model.enable_input_require_grads()
else:
    print("Full fine-tuning mode (no LoRA)")

base_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

trainable = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in base_model.parameters())
print(f"Trainable: {trainable:,} / {total_p:,} ({100*trainable/total_p:.2f}%)")
print("✓ Fresh model loaded for baseline SFT")



In [ ]:
# ============================================================
# BASELINE TRAINING LOOP — CE ONLY, 1 forward pass / sample
# Same hyperparams, same data, same schedule
# ============================================================
print("=" * 70)
print("  STARTING BASELINE SFT (CE only, no JEPA)")
print("=" * 70)

BASELINE_RESULTS_DIR = "results_baseline_sft"
os.makedirs(BASELINE_RESULTS_DIR, exist_ok=True)

base_model.train()

bl_optimizer = Adafactor(
    filter(lambda p: p.requires_grad, base_model.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    relative_step=False,
    scale_parameter=False,
    warmup_init=False,
)

bl_scheduler = get_linear_schedule_with_warmup(
    bl_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

bl_training_log = []
bl_global_step  = 0
bl_accumulated  = 0
bl_optimizer.zero_grad()
bl_total_start  = time.time()
bl_sample_times = []

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    epoch_ce = []

    epoch_samples = train_samples.copy()
    random.shuffle(epoch_samples)

    print(f"\n{'='*70}")
    print(f"  EPOCH {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*70}")

    for i, sample in enumerate(epoch_samples):
        t0 = time.time()

        # ---- BUILD PROMPT (reuse existing helper) ----
        pa_full_text, pa_q_text = build_prompts(sample)
        pa_q_len = len(tokenizer.encode(pa_q_text))

        # ---- TOKENISE ----
        pa_enc = tokenizer(pa_full_text, return_tensors="pt",
                           truncation=True, max_length=MAX_LENGTH).to(device)

        # ---- LABELS (mask question, keep answer — same as JEPA) ----
        labels = pa_enc["input_ids"].clone()
        mask_end = min(pa_q_len, labels.shape[1])
        labels[0, :mask_end] = -100

        # =========================================================
        # SINGLE FORWARD PASS — CE loss only (no English, no JEPA)
        # =========================================================
        outputs = base_model(
            input_ids=pa_enc["input_ids"],
            attention_mask=pa_enc["attention_mask"],
            labels=labels,
        )
        ce_loss = outputs.loss

        # ---- BACKWARD ----
        (ce_loss / GRAD_ACCUM_STEPS).backward()
        bl_accumulated += 1

        if bl_accumulated >= GRAD_ACCUM_STEPS:
            torch.nn.utils.clip_grad_norm_(base_model.parameters(), MAX_GRAD_NORM)
            bl_optimizer.step()
            bl_scheduler.step()
            bl_optimizer.zero_grad()
            bl_accumulated = 0
            bl_global_step += 1

        # ---- RECORD ----
        dt = time.time() - t0
        bl_sample_times.append(dt)
        cv = ce_loss.item()
        epoch_ce.append(cv)

        bl_training_log.append({
            "epoch": epoch + 1, "sample": i + 1,
            "global_sample": epoch * NUM_TRAIN + i + 1,
            "global_step": bl_global_step,
            "ce_loss": round(cv, 6),
            "lr": bl_scheduler.get_last_lr()[0],
            "step_time_s": round(dt, 3),
        })

        if (i + 1) % 50 == 0 or i == 0 or (i + 1) == NUM_TRAIN:
            window = min(50, len(epoch_ce))
            avg_t = sum(bl_sample_times[-window:]) / window
            rem   = (NUM_EPOCHS - epoch) * NUM_TRAIN - (i + 1)
            eta   = str(timedelta(seconds=int(rem * avg_t)))
            print(
                f"  [{i+1:3d}/{NUM_TRAIN}] "
                f"CE:{sum(epoch_ce[-window:])/window:.4f}  "
                f"LR:{bl_scheduler.get_last_lr()[0]:.1e}  "
                f"{avg_t:.2f}s/samp  ETA:{eta}"
            )

        # ---- SAVE CHECKPOINT (EVERY 100 SAMPLES) ----
        if (i + 1) % 100 == 0:
            step_ckpt_path = f"{BASELINE_RESULTS_DIR}/checkpoint_epoch_{epoch+1}_step_{i+1}"
            print(f"\n  Saving step checkpoint to {step_ckpt_path}...")
            base_model.save_pretrained(step_ckpt_path)
            tokenizer.save_pretrained(step_ckpt_path)
            
        del outputs, ce_loss
        torch.cuda.empty_cache()

    # Flush remaining
    if bl_accumulated > 0:
        torch.nn.utils.clip_grad_norm_(base_model.parameters(), MAX_GRAD_NORM)
        bl_optimizer.step(); bl_scheduler.step(); bl_optimizer.zero_grad()
        bl_accumulated = 0; bl_global_step += 1

    et = time.time() - epoch_start
    print(f"\n  ── Epoch {epoch+1} summary ──")
    print(f"     CE Loss   : {sum(epoch_ce)/len(epoch_ce):.4f}")
    print(f"     Time      : {et:.0f}s  ({et/60:.1f} min)")
    print(f"     Opt. steps: {bl_global_step}")

    # ---- SAVE CHECKPOINT (END OF EPOCH) ----
    epoch_ckpt_path = f"{BASELINE_RESULTS_DIR}/checkpoint_epoch_{epoch+1}"
    print(f"\n  Saving epoch checkpoint to {epoch_ckpt_path}...")
    base_model.save_pretrained(epoch_ckpt_path)
    tokenizer.save_pretrained(epoch_ckpt_path)

bl_tt = time.time() - bl_total_start
print(f"\n{'='*70}")
print(f"  BASELINE TRAINING COMPLETE — {bl_tt:.0f}s ({bl_tt/60:.1f} min)")
print(f"  Total optimizer steps: {bl_global_step}")
print(f"{'='*70}")


In [ ]:
# ============================================================
# BASELINE EVALUATION — same 100 unseen samples
# ============================================================
print("=" * 70)
print("  BASELINE EVALUATION: 100 unseen Punjabi GSM8K samples")
print("=" * 70)

base_model.gradient_checkpointing_disable()
base_model.eval()

bl_eval_logs = []
bl_correct   = 0
bl_eval_start = time.time()

for i, sample in enumerate(test_samples):
    question = sample["pa_question"]
    gold_raw = sample["pa_answer"]
    gold     = extract_gold_answer(gold_raw)
    if gold is None:
        gold = extract_gold_answer(sample["en_answer"])

    prompt = PROMPT_TEMPLATE.format(question=question)
    msgs   = [{"role": "user", "content": prompt}]
    text   = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(text, return_tensors="pt",
                       truncation=True, max_length=MAX_LENGTH).to(device)

    with torch.no_grad():
        gen = base_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.7, top_p=0.8, top_k=20,
            do_sample=True,
        )
    out_ids  = gen[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(out_ids, skip_special_tokens=True).strip()
    pred     = extract_model_answer(response)
    hit      = compare_answers(pred, gold)
    if hit:
        bl_correct += 1

    bl_eval_logs.append({
        "index": i, "pa_question": question,
        "en_question": sample["en_question"],
        "gold": gold, "pred": pred, "correct": hit,
        "response_preview": response[:400],
    })

    if (i + 1) % 10 == 0 or i == 0:
        print(f"  [{i+1:3d}/{NUM_TEST}]  "
              f"Gold:{str(gold):>6s}  Pred:{str(pred):>6s}  "
              f"{'✓' if hit else '✗'}   "
              f"Running: {bl_correct}/{i+1} = {bl_correct/(i+1)*100:.1f}%")

bl_eval_time = time.time() - bl_eval_start
bl_accuracy  = bl_correct / NUM_TEST * 100

print(f"\n{'='*70}")
print(f"  BASELINE RESULT: {bl_correct}/{NUM_TEST} = {bl_accuracy:.1f}%")
print(f"  Time: {bl_eval_time:.0f}s  ({bl_eval_time/NUM_TEST:.1f}s / sample)")
print(f"{'='*70}")


In [ ]:
# ============================================================
# SAVE BASELINE & COMPARE WITH JEPA
# ============================================================
bl_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

bl_summary = {
    "model": MODEL_NAME,
    "method": "Baseline SFT (CE only, no JEPA)",
    "language": f"{LANG_NAME} ({LANG_CONFIG})",
    "timestamp": bl_timestamp,
    "config": {
        "seed": SEED, "lr": LEARNING_RATE, "epochs": NUM_EPOCHS,
        "grad_accum": GRAD_ACCUM_STEPS, "max_length": MAX_LENGTH,
        "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
        "loss": "CE only",
        "num_train": NUM_TRAIN, "num_test": NUM_TEST,
        "forward_passes_per_sample": 1,
    },
    "eval": {
        "correct": bl_correct, "total": NUM_TEST,
        "accuracy": round(bl_accuracy, 2),
        "time_s": round(bl_eval_time, 1),
    },
    "training_time_s": round(bl_tt, 1),
}

bl_all_data = {
    "summary": bl_summary,
    "training_log": bl_training_log,
    "eval_logs": bl_eval_logs,
}

bl_log_path = f"{BASELINE_RESULTS_DIR}/baseline_sft_train_{bl_timestamp}.json"
bl_sum_path = f"{BASELINE_RESULTS_DIR}/baseline_sft_summary_{bl_timestamp}.json"

with open(bl_log_path, "w", encoding="utf-8") as f:
    json.dump(bl_all_data, f, ensure_ascii=False, indent=2)
with open(bl_sum_path, "w", encoding="utf-8") as f:
    json.dump(bl_summary, f, ensure_ascii=False, indent=2)

# Save baseline LoRA adapter
bl_adapter_path = f"{BASELINE_RESULTS_DIR}/lora_adapter_{bl_timestamp}"
base_model.save_pretrained(bl_adapter_path)
tokenizer.save_pretrained(bl_adapter_path)

print(f"Baseline logs    → {bl_log_path}")
print(f"Baseline summary → {bl_sum_path}")
print(f"Baseline adapter → {bl_adapter_path}/")

# ============================================================
# HEAD-TO-HEAD COMPARISON
# ============================================================

cv_accuracy = accuracy   # from concept-vec SFT eval
cv_time     = tt         # from concept-vec SFT training

print(f"\n{'='*70}")
print(f"  HEAD-TO-HEAD: ConceptVec SFT vs Baseline SFT")
print(f"{'='*70}")
print(f"  {'':30s} {'ConceptVec':>12s} {'Baseline':>12s} {'Δ':>8s}")
print(f"  {'-'*64}")
print(f"  {'Eval Accuracy (%)':30s} {cv_accuracy:>11.1f}% {bl_accuracy:>11.1f}% {bl_accuracy - cv_accuracy:>+7.1f}%")
print(f"  {'Training Time (min)':30s} {cv_time/60:>11.1f}  {bl_tt/60:>11.1f}  {bl_tt/60 - cv_time/60:>+7.1f}")
print(f"  {'Loss':30s} {'CE+CosSim':>12s} {'CE only':>12s}")
print(f"  {'CosSim Δ (post−pre)':30s} {post_cos_sim - baseline_cos_sim:>+11.4f}  {'N/A':>12s}")
print(f"{'='*70}")
